# hist

> Converting dialogs into LLM chat history

In [ ]:
#| default_exp hist

## Imports

In [ ]:
#| export
import re, base64, binascii, hashlib
from ast import literal_eval
from datetime import datetime
from zoneinfo import ZoneInfo
from itertools import chain

from fastcore.utils import *
from fastcore.ansi import strip_ansi
from fastcore.xtras import detect_mime
from fastcore.nbio import preferred_msg_out, concat_streams
from fastcore.xml import to_xml, Media, MediaUnavailable, Output, CellSource, Message as XMLMessage
from fastllm.chat import re_token, re_tools, MediaUrl, mk_msg, fmt2hist, Msg
from fastllm.types import PartType
from llmsurgery.dialog import *

In [ ]:
from fastllm.chat import mk_msgs, mk_msg, tool_dtls_tag
from fastcore.test import *
import random, json
from IPython.display import Markdown

In [ ]:
def tool():
    "dummmy tool"
    pass

In [ ]:
random.seed(6)

### Overview

| Function | Purpose | Input → Output |
|----------|---------|----------------|
| `render_outputs_ai` | Render Jupyter outputs as plain text | `list[dict]` → `str` |
| `ai_fmt` | Format an AI reply for chat history | `str` → `str` |
| `Message.to_xml` | Convert message to XML | `Message` → `str` |
| `Message.to_media` | Gather media from message | `Message` → `list[str\|bytes]` |
| `Message.to_parts` | Convert message to history parts | `Message` → `list[str\|bytes]` |
| `msgs2hist` | **Main entry point** | `list[Message], dict` → `hist` |

## Test helpers

`mk` is a convenience function to create conformant dummy data for testing. For `msg_type=='code'` cells, output text is a serialized structure following nbformat v4, following solveit. For creating tests, we have a helper function `jwrap` that wraps output text in this jupyter v4 format.

In [ ]:
dname = 'backend'
dlg = Dialog(dname)

In [ ]:
def jwrap(txts:list=['']):
    "Wrap plain `txt` in a minimal Jupyter-style cell output"
    return [{'output_type':'stream','name':'stdout','text':txt} for txt in listify(txts)]

def mk(msg_type: str, content=None, output='', id=None, attachments=None):
    "Return a `Message`, ensuring `output` is always valid JSON. Uses timestamp-based id if not provided."
    if msg_type==scode: output = jwrap(output)
    if msg_type==sprompt: output = prompt_output(output)
    res = Message(msg_type=msg_type, output=output, attachments=attachments)
    res.content = content or f'{msg_type}>{res.id}'
    res.dlg = dlg # to avoid weakref issue
    return res

In [ ]:
jwrap("test output")

[{'output_type': 'stream', 'name': 'stdout', 'text': 'test output'}]

Code cells automatically `jwrap` the output value:

In [ ]:
mk('code', 'print("hello")', 'hello\n')

<div class="prose" markdown="1">

print("hello") ⇒ [{'output_type': 'stream', 'name': 'stdout', 'text': 'hello\n'}]

<details>

- id: _29f88512
- time_run: 
- is_exported: False
- skipped: False
- bookmark: None
- i_collapsed: False
- o_collapsed: False
- heading_collapsed: False
- i_clamp: False
- o_clamp: False
- pinned: False
- run: None
- msg_type: code
- input_tokens: None
- output_tokens: None

</details>

</div>

In [ ]:
mk('prompt', 'What is 2+2?', 'The answer is 4')

<div class="prose" markdown="1">

What is 2+2? ⇒ [{'output_type': 'display_data', 'metadata': {'is_ai_res': True}, 'data': {'text/markdown': 'The answer is 4'}}]

<details>

- id: _004af0bf
- time_run: 
- is_exported: False
- skipped: False
- bookmark: None
- i_collapsed: False
- o_collapsed: False
- heading_collapsed: False
- i_clamp: False
- o_clamp: False
- pinned: False
- run: None
- msg_type: prompt
- input_tokens: None
- output_tokens: None

</details>

</div>

## Rendering outputs for the AI

The model sees a plain-text rendering of Jupyter outputs: streams concatenated as they would appear on a terminal screen, ANSI codes stripped, markdown preferred over HTML, images excluded (they travel separately, as media parts). The Jupyter-output primitives — `preferred_out`, `preferred_msg_out`, `concat_streams`, and the screen-semantics cleaning they build on — live in `fastcore.nbio` and `fastcore.xtras`; this module only adds the choices specific to model-facing text.

In [ ]:
#| export
def join_out(d):
    "Join Jupyter's list-of-lines output data into one string"
    return ''.join(d) if isinstance(d, list) else d

AI_RENDERERS = {
    'text/plain': lambda d: strip_ansi(str(d)),
    'text/html': join_out,
    'text/markdown': join_out,
    'text/latex': join_out,
    'application/javascript': lambda d: f'<script>{join_out(d)}</script>',
}

def render_output_ai(out, renderers=None):
    "Plain-text rendering of one Jupyter output, as the AI sees it; `renderers` overrides/extends the per-mime table"
    r = AI_RENDERERS if renderers is None else AI_RENDERERS|renderers
    ptyp,d = preferred_msg_out(out, html1st=False, include_imgs=False)
    return r.get(ptyp, lambda d: '')(d)

def render_outputs_ai(outputs, renderers=None):
    "Plain-text rendering of a Jupyter output list for LLM context"
    if (not isinstance(outputs, (list,tuple))) or (outputs and not isinstance(outputs[0],dict)):
        print(f"Unexpected outputs: {outputs}")
        return ''
    return '\n'.join(render_output_ai(o, renderers=renderers) for o in concat_streams(outputs))

Each output type has a sensible plain-text form; note the stream applying the `\r` overwrite, and the error rendering its traceback:

In [ ]:
test_eq(render_outputs_ai([{'output_type': 'stream', 'name': 'stdout', 'text': 'Hello\rBye\n'}]), 'Bye\n')
test_eq(render_output_ai({'output_type': 'execute_result', 'data': {'text/plain': '"42"'}}), '"42"')
test_eq(render_output_ai({'output_type': 'display_data', 'data': {'text/markdown': '**bold**'}}), '**bold**')
test_eq(render_output_ai({'output_type': 'display_data', 'data': {'text/latex': '$x^2$'}}), '$x^2$')  # verbatim by default
test_eq(render_output_ai({'output_type': 'display_data', 'data': {'text/latex': '$x^2$'}}, renderers={'text/latex': str.upper}), '$X^2$')
test_eq(render_output_ai({'output_type': 'error', 'traceback': ['Error: something broke']}), 'Error: something broke')
test_eq(render_output_ai({'output_type': 'display_data', 'data': {'text/html': '<b>hi</b>', 'text/plain': 'hi'}}), '<b>hi</b>')
assert not render_output_ai({'output_type': 'execute_result', 'data': {'text/plain': ''}})

## Reply formatting

In [ ]:
#| export
def ai_fmt(out, strip_tools=False):
    "Format AI response as it will be attached to the chat history"
    res = out
    if strip_tools:
        res = re_token.sub('', res)
        res = re_tools.sub('', res)
    res = res.strip().removesuffix("\n*[Response interrupted]*").strip()
    return res or '<output result="pending" reason="incomplete"/>'

Interrupted-response markers are stripped, and an empty reply becomes an explicit pending placeholder so the history never carries a blank turn. `strip_tools` also removes embedded tool-call and token-usage details blocks (the fastllm reply conventions).

In [ ]:
test_eq(ai_fmt('Hi.\n*[Response interrupted]*'), 'Hi.')
test_eq(ai_fmt(''), '<output result="pending" reason="incomplete"/>')
dtl = f"{tool_dtls_tag}\n\n```json\n{{}}\n```\n\n</details>"
test_eq(ai_fmt(f'{dtl}\nDone.', strip_tools=True), 'Done.')

## Message AI views

In [ ]:
#| export
_prims = str, int, float, complex, bool, tuple, list, dict, set, frozenset

def try_eval(s, typ:str|None=None):
    "Like `literal_eval`, but wraps in dynamic class named `typ` if succeeds, and returns `s` if fails"
    try:
        res = literal_eval(s)
        if typ and isinstance(res, _prims): res = type(typ, (type(res),), {})(res)
        return res
    except: return s

`ai_output` is the model-facing text of a message's output, computed through `render_out` and cached in `_ai_rend`. `render_out` computes only the AI side; a UI (like Solveit) overrides it on its `Message` subclass to compute and cache its HTML rendering too, exposed through properties of its own.

In [ ]:
#| export
Message.ai_renderers = {}

@patch
def render_out(self:Message):
    "Compute and cache the for-AI rendering of this message's output"
    if self._ai_rend is not None: return
    if self.msg_type in (sprompt, scode):
        air = render_outputs_ai(self.output or [], renderers=self.ai_renderers)
        self._ai_rend = ai_fmt(air) if self.msg_type == sprompt else str(try_eval(air))
    else: self._ai_rend = ''

@patch(as_prop=True)
def ai_output(self:Message):
    self.render_out()
    return self._ai_rend

In [ ]:
test_eq(Message('1+1', msg_type=scode, output=code_output('2')).ai_output, '2')
test_eq(Message('1+1', msg_type=scode).ai_output, '')  # No output yet
m = Message('test', msg_type=sprompt, output=prompt_output('Ok!\n\n*[Response interrupted]*\n'))
test_eq(m.ai_output, 'Ok!')
test_eq(Message('empty', msg_type=sprompt, output=[]).ai_output, '<output result="pending" reason="incomplete"/>')

In [ ]:
#| export
def to_local_time(time_str, tz='UTC'):
    "Convert ISO time string to local timezone"
    if not time_str: return ''
    try: dt = datetime.fromisoformat(time_str)
    except ValueError: return ''
    return dt.astimezone(ZoneInfo(tz)).isoformat()

@patch
def local_time(self:Message):
    "Localized `time_run` for hosts that stamp one; '' when absent"
    if not (t := getattr(self, 'time_run', '')): return ''
    tz = getattr(self.dlg, 'timezone', 'UTC') if self.dlg else 'UTC'
    return to_local_time(t, tz)

In [ ]:
test_eq(to_local_time(''), '')
test_eq(to_local_time(None), '')
result = to_local_time('2026-01-27T12:00:00+00:00', 'UTC')
assert '2026-01-27' in result

m = Message('test', msg_type=snote)
test_eq(m.local_time(), '')

m.time_run = '2026-01-27T12:00:00+00:00'
assert m.local_time()

In [ ]:
#| export
@patch
def todict(self:Message):
    res = AttrDict({k:v for k,v in asdict(self).items() if k[0]!='_' and k!='attachments'})
    res['output'] = self.ai_output
    return res

## Media

Media in the LLM context is represented by an XML metadata tag followed by raw bytes.

For successfully loaded media, the context contains:

- a `<media>` tag with `id`, `type`, and `mime` attributes
- the raw media bytes immediately after the tag

If media cannot be loaded or decoded, the context contains a `<media-unavailable>` tag instead of silently dropping it. This lets the AI tell the user that the media was referenced but was not available.

`id=` is:

- UUID for attachments referenced via `attachment:uuid` in markdown
- variable name for media in variables
- content hash for code output images

`type=` is one of:

- `output` — media from code cell output
- `variable` — media data in a variable
- `content` — media embedded in markdown message content

### Media context tags

Media are included via a `<media>` tag followed immediately by the raw bytes. The tag gives the AI a stable id, media type, and MIME type.

For unavailable media, we add a `<media-unavailable>` tag instead. This is used when media was referenced by the user but could not be loaded, decoded, resized, or safely resolved. The tags instructs the AI to tell the user that it could not see the referenced file.


Model info has the following fields `supports_vision`, `supports_video_input`, `supports_pdf_input`, `supports_audio_input` to indicate specific media type support

In [ ]:
#| export
def _mime_kind(mime):
    if mime == 'application/pdf': return 'pdf'
    return mime.split('/', 1)[0] if mime and mime.split('/', 1)[0] in {'image','audio','video'} else None

def _mime_supported(mime, aim_info):
    if   _mime_kind(mime) == 'image' and aim_info.get('supports_vision', False):      return True
    elif _mime_kind(mime) == 'video' and aim_info.get('supports_video_input', False): return True
    elif _mime_kind(mime) == 'audio' and aim_info.get('supports_audio_input', False): return True
    elif _mime_kind(mime) == 'pdf'   and aim_info.get('supports_pdf_input', False):   return True
    return False

In [ ]:
#| export
class _MissError(Exception): pass

UNSUPPORTED_MSG = "An unsupported media type was included in the context: {mime}. Please instruct the user to switch to a model that supports this media type."
Message.UNSUPPORTED_MSG = UNSUPPORTED_MSG

def _media_xml(media_type, media_id, mime=None):
    "Create a media XML tag for LLM context."
    return to_xml(Media(id=media_id, type=media_type, mime=mime))

def _media_unavailable(media_type, media_id, e, mime=None):
    "Create a media-unavailable XML tag for LLM context."
    msg = "The user included this media, but it was not available to the AI. Tell the user that you cannot see it."
    return to_xml(MediaUnavailable(msg, id=str(media_id), type=media_type, reason=str(e), mime=mime))

def media_item(media_type, media_id, data, aim_info, mime=None, max_im_sz=None, prep=None, unavail_msg=UNSUPPORTED_MSG):
    "Load/prepare/package media, returning [xml_tag, data] or [media-unavailable]."
    try:
        # `data` can be bytes, or a function that loads bytes lazily.
        data = data() if callable(data) else data
        if data is None: raise _MissError("media data unavailable")
        if not isinstance(data, (bytes, MediaUrl)): raise _MissError(f"expected bytes or MediaUrl, got {type(data).__name__}")
        mime = mime or (detect_mime(data) if isinstance(data, bytes) else data.mime)
        if not mime: raise _MissError("unsupported or unknown media type")
        if not _mime_supported(mime, aim_info): raise _MissError(unavail_msg.format(mime=mime))
        # `media_id` can be a string, or a function that derives an id.
        media_id = media_id(data) if callable(media_id) else media_id
        if prep and isinstance(data, bytes): data = prep(data, mime, max_im_sz)
        return [_media_xml(media_type, media_id, mime), data]
    except (_MissError, OSError, binascii.Error) as e:
        media_id = getattr(media_id, '__name__', 'unknown') if callable(media_id) else media_id
        return [_media_unavailable(media_type, media_id, e, mime)]

In [ ]:
detect_mime('https://image.jpg')

A helper function and small dummy img for testing purposes.

In [ ]:
def _mk_img(imgb):
    "Image bytes from a base64 str"
    return base64.b64decode(imgb)

In [ ]:
_imgb64 = b'iVBORw0KGgoAAAANSUhEUgAAAAEAAAABCAAAAAA6fptVAAAACklEQVR4nGP4DwABAQEAsTj2FAAAAABJRU5ErkJggg=='

Valid media should produce a `<media>` tag plus bytes

In [ ]:
png = _mk_img(_imgb64)
res = media_item('content', 'img', png, dict(supports_vision=True))
test_eq(res[0], '<media id="img" type="content" mime="image/png"></media>')
test(res[1], bytes, isinstance)

Expected media-loading failures should produce `<media-unavailable>`

In [ ]:
for data in (None, 'not bytes', b'not media'):
    test(media_item('content', 'x', data, {})[0], '<media-unavailable', operator.contains)

def bad_b64_loader(): raise binascii.Error('bad base64')
test(media_item('output', 'bad', bad_b64_loader, {})[0], '<media-unavailable', operator.contains)

Unexpected programmer errors should still propagate.

In [ ]:
def bad_loader(): raise RuntimeError('boom')
with expect_fail(Exception, 'boom'): media_item('content', 'x', bad_loader, {})

Unsupported media types by AI return `<media-unavailable>`

In [ ]:
test(media_item('content', 'img', png, dict(supports_vision=False))[0], "An unsupported media type was", operator.contains)
test(media_item('content', 'img', png, dict(supports_vision=False))[0], 'mime="image/png"', operator.contains)

### `_media_atts`

These are attachments found in a message such as the ones uploaded from the clipboard or the screenshot of the last output.

In [ ]:
#| export
@patch
def prep_img(self:Message, data, mime, max_im_sz=None):
    "Prepare image bytes before they enter LLM context; the base class passes them through unchanged"
    return data

In [ ]:
#| export
def _media_atts(msg, aim_info, max_im_sz=None):
    "Build media context from a message's attachments that are referenced in the content."
    attids = set()
    attids.update(re.findall(r'!\[[^\]]*\]\(attachment:([0-9a-f-]{36})\)', msg.content))
    attids.update(re.findall(r'<!-- last_output:([0-9a-f-]{36}) -->', msg.content))
    atts = L(msg.attachments).filter(lambda o: o in sorted(attids))
    return list(chain.from_iterable(media_item('content', o.id, o.data, aim_info=aim_info, max_im_sz=max_im_sz,
        prep=msg.prep_img, unavail_msg=msg.UNSUPPORTED_MSG) for o in atts))

In [ ]:
def parse_att_id(s):
    "Extract id attribute from XML tag string"
    return re.search(r' id="([^"]*)"', s).group(1)

In [ ]:
_mk_img(_imgb64)[:40]

b'\x89PNG\r\n\x1a\n\x00\x00\x00\rIHDR\x00\x00\x00\x01\x00\x00\x00\x01\x08\x00\x00\x00\x00:~\x9bU\x00\x00\x00\nIDA'

Only valid image attachments should be included:

In [ ]:
att1,att2,att3 = Attachment(png, 'image/png'), Attachment(b'not an image', 'text/plain'), Attachment(_mk_img(_imgb64), 'image/png')
m = Message(f'![](attachment:{att1.id}) and ![](attachment:{att3.id})', attachments=[att1, att2, att3])
res = _media_atts(m, dict(supports_vision=True))

In [ ]:
test_eq(len(m.attachments), 3)
test_eq(len(res), 4)
test_eq(L(res[::2]).map(parse_att_id), [att1.id, att3.id])

Only the attachments referenced in the message content should be included:

In [ ]:
m = Message(f'Only ![](attachment:{att3.id})', attachments=[att1,att3])
res = _media_atts(m, dict(supports_vision=True))

In [ ]:
test_eq(len(m.attachments), 2)
test_eq(L(res[::2]).map(parse_att_id), [att3.id])

In [ ]:
bad = Attachment(b'not an image', 'image/png')
res = _media_atts(Message(f'![](attachment:{bad.id})', attachments=[bad]), dict(supports_vision=True))
test(res[0], '<media-unavailable', operator.contains)

### `_img_output`

These are images from the message outputs, such as a code output displaying a plot

In [ ]:
#| export
def _img_id(im_bytes):
    "Generate a deterministic 8 char id from `im_bytes`"
    h = hashlib.md5(im_bytes)
    return base64.b32encode(h.digest()).decode('utf-8')[:8]

In [ ]:
test_eq(_img_id(_mk_img(_imgb64)), _img_id(_mk_img(_imgb64)))
_img_id(_mk_img(_imgb64))

'4P52II3F'

In [ ]:
#| export
def _img_output(m, aim_info, max_im_sz):
    "Extract any image outputs from `m` and return a list of encoded images and their xml tags."
    outs = []
    for o in m.output or []:
        for mime, data in o.get('data', {}).items():
            if mime in {'image/jpeg', 'image/png'}:
                outs += media_item('output', _img_id, lambda data=data: base64.b64decode(data), aim_info, mime,
                    max_im_sz, prep=m.prep_img, unavail_msg=m.UNSUPPORTED_MSG)
    return outs

In [ ]:
img_out = mk_code_output({'image/png': base64.b64encode(png).decode()})
m = Message(msg_type='code', output=img_out)

In [ ]:
res = _img_output(m, dict(supports_vision=True), None)
test_eq(len(res), 2)
test_eq(res[0], _media_xml('output', _img_id(png), 'image/png'))

In [ ]:
m = Message(msg_type='code', output=mk_code_output({'image/png': 'not-base64'}))
res = _img_output(m, dict(supports_vision=True), None)
test(res[0], '<media-unavailable', operator.contains)

## Message XML Helpers

These methods convert `Message` objects into XML elements and gather associated media for LLM context.

`_xml_content` generates the cell contents for non-prompts or those in `expr_mtypes`

In [ ]:
#| export
@patch
def _xml_content(self:Message):
    "Get content elements for XML"
    return [CellSource(self.content)]

In [ ]:
# Returns content for all message types
test_eq(mk('note', 'hello')._xml_content(), [CellSource('hello')])
test_eq(mk('code', 'x=1')._xml_content(), [CellSource('x=1')])
test_eq(mk('prompt', 'question')._xml_content(), [CellSource('question')])

`to_xml` combines the msg input and output. A prompt renders through `prompt_txt` — bare content here, with `last` marking the live request; a host that wants an envelope around its prompts (as solveit does) patches `prompt_txt` alone:

In [ ]:
#| export
@patch
def prompt_txt(self:Message, last=False):
    "History text for a prompt message: the content, bare. Hosts patch this to add their own envelope"
    return self.content

@patch
def to_xml(self:Message, last=False):
    "Create XML representation of this message"
    if self.msg_type == 'prompt': return self.prompt_txt(last)
    elems = [CellSource(self.content) if self.content else "", Output(self.ai_output) if self.ai_output else ""]
    return to_xml(XMLMessage(*elems, type=self.msg_type, id=self.id, time=self.local_time()), do_escape=False)

In [ ]:
# Combines content and output into full XML representation
m = mk('code', 'print("hi")', 'hi\n')
m.to_xml()

'<message type="code" id="_3acc45ed" time=""><cell-source>print("hi")</cell-source><output>hi\n</output></message>'

In [ ]:
# Prompts render bare
test_eq(mk('prompt', 'question').to_xml(), 'question')
test_eq(mk('prompt', 'question').prompt_txt(last=True), 'question')

In [ ]:
#| export
@patch
def media_extra(self:Message, aim_info, max_im_sz=None):
    "Additional media sources contributed by subclasses (e.g. app-specific file references)"
    return []

@patch
def to_media(self:Message, aim_info, max_im_sz=None):
    "Aggregate media from attachments, `media_extra`, and code outputs"
    media_ctx = []
    if media_atts := _media_atts(self, aim_info, max_im_sz): media_ctx += media_atts
    if extra := self.media_extra(aim_info, max_im_sz): media_ctx += extra
    if self.msg_type == scode:
        if img_outputs := _img_output(self, aim_info, max_im_sz): media_ctx += img_outputs
    return media_ctx

In [ ]:
# No media for plain messages
test_eq(mk('note', 'hello').to_media({}), [])

In [ ]:
# Code output images
m = Message(msg_type='code', output=mk_code_output({'image/png': base64.b64encode(png).decode()}))
media = m.to_media(dict(supports_vision=True))
test_eq(len(media), 2)  # xml tag, bytes

## `.to_parts`

In [ ]:
#| export
@patch
def to_parts(self:Message, aim_info:dict, last=False):
    "Convert message to a list of history parts: media, then XML text."
    parts = []
    if media := self.to_media(aim_info): parts.extend(media)
    if mxml := self.to_xml(last): parts.append(mxml)
    return parts

In [ ]:
att = Attachment(_mk_img(_imgb64), 'image/png')
m = mk('prompt', f'What is in ![](attachment:{att.id})?', attachments=[att])
res = m.to_parts(dict(supports_vision=True))
['...bytes...' if isinstance(o,bytes) else o for o in res]

['<media id="puppy.jpg" type="content" mime="image/jpeg"></media>',
 '...bytes...',
 '<instructions>Respond to the request in the `prompt` below.</instructions>\n<prompt sid="_77be8b5e">Hi $`name`. What is in ![](puppy.jpg#ai)? What is in $`img`?</prompt>']

## `msgs2hist`

In [ ]:
#| export
def msgs2hist(msgs:list[Message], aim_info:dict):
    "Convert messages to LLM history. `msgs` must end with a prompt, which renders with `last=True`."
    if msgs[-1].msg_type != sprompt: raise ValueError("msgs2hist requires msgs to end with a prompt")
    res = []
    for is_first, m in loop_first(reversed(msgs)):
        if m.msg_type == sprompt: res += [m.ai_output, m.to_parts(aim_info, last=is_first)]
        else: res[-1] = m.to_parts(aim_info) + res[-1]
    return res[::-1]

## Dialogs to canonical messages

`msgs2hist` produces alternating user parts and AI reply strings. For surgery we usually want the replies' tool calls back as structured messages, and fastllm's `fmt2hist` inverts the reply format exactly. `dlg2canon` composes the two into the canonical form every provider conversion starts from, and rejects duplicate tool-call ids, which hand-authored dialogs can accidentally produce.

In [ ]:
#| export
def _seq_tools(msgs):
    "Explode batched tool calls into strict call/result alternation"
    out = []
    for m in msgs:
        if m.role=='tool' and out and out[-1].role=='assistant':
            a = out.pop()
            tus = [p for p in a.content if p.type==PartType.tool_use]
            if len(tus)>1:
                pre,trs = [p for p in a.content if p.type!=PartType.tool_use],{p.data['id']:p for p in m.content}
                for tu in tus:
                    out.append(Msg(role='assistant', content=pre+[tu]))
                    out.append(Msg(role='tool', content=[trs[tu.data['id']]]))
                    pre = []
                continue
            out.append(a)
        out.append(m)
    return out

def dlg2canon(
    dlg, # A `Dialog`, ending with a prompt
    aim_info=None, # Model capability dict for media handling; images enabled if None
):
    "Canonical fastllm messages for `dlg`, with each reply's tool calls recovered as real parts"
    hist = msgs2hist(dlg.messages, dict(supports_vision=True) if aim_info is None else aim_info)
    msgs = []
    for i,t in enumerate(hist):
        if i%2==0: msgs.append(mk_msg(t))
        else: msgs += _seq_tools(fmt2hist(t))
    ids = [p.data['id'] for m in msgs for p in m.content if p.type==PartType.tool_use]
    if dups := {i for i in ids if ids.count(i)>1}: raise ValueError(f"duplicate tool call id(s): {', '.join(sorted(dups))}")
    return msgs

The conversion tests below need replies carrying tool-call details blocks, so first a fixture that builds one in fastllm's reply format.

In [ ]:
def _mk_dtl(func, args, result, tid='x1'):
    d = json.dumps({'id':tid, 'server':False, 'call':{'function':func, 'arguments':args}, 'result':result})
    return f"{tool_dtls_tag}\n<summary><code>{func}(...)</code></summary>\n\n```json\n{d}\n```\n\n</details>"

py_dtl = _mk_dtl('python', {'code':'1+1'}, '2')

In [ ]:
ddlg = Dialog('canon')
ddlg.mk_message('What is 1+1?', msg_type=sprompt, output=f"Check:\n\n{py_dtl}\n\nIt is 2.")
cm = dlg2canon(ddlg)
test_eq([m.role for m in cm], ['user', 'assistant', 'tool', 'assistant'])
test_eq(cm[1].content[1].data['name'], 'python')

One reply often holds several sequential tool calls, each issued after reading the previous result. `fmt2hist` returns them batched into a single call message and a single result message, which reads as one parallel volley; `dlg2canon` explodes the batch back into strict call/result alternation, the shape live sessions record.

In [ ]:
py_dtl2 = _mk_dtl('python', {'code':'2+2'}, '4', 'x2')
sdlg = Dialog('seq')
sdlg.mk_message('Two steps.', msg_type=sprompt, output=f"{py_dtl}\n\n{py_dtl2}\n\nDone.")
sm = dlg2canon(sdlg)
test_eq([(m.role, len(m.content)) for m in sm], [('user',1), ('assistant',1), ('tool',1), ('assistant',1), ('tool',1), ('assistant',1)])
test_eq(sm[2].content[0].data['id'], sm[1].content[0].data['id'])
test_eq(sm[4].content[0].data['id'], sm[3].content[0].data['id'])

In [ ]:
bad = Dialog('dup')
bad.mk_message('a', msg_type=sprompt, output=py_dtl)
bad.mk_message('b', msg_type=sprompt, output=py_dtl)
with expect_fail(ValueError, 'duplicate tool call id'): dlg2canon(bad)

In [ ]:
raw = [mk('prompt', 'hello $`x`', output='Hello! I can see the variable `x`'),
       mk('note', '$`not_variable1`'), 
       mk('code', output="aaa", content="'$`not_variable2`'"), 
       mk('prompt', 'hi hi $`y`', output='Hello! I can see the variable `y`'),
       mk('note'), 
       mk('note'), 
       mk('code', output="333"),
       mk('prompt'),  # empty output on purpose
       mk('prompt', '$`x`, $`y`', output='Hello! I can see the variables `x` and `y`'),
       mk('note', 'below we insert an image'),
       mk('code', 'img=..some code to read image bytes'),
       mk('prompt', 'Please tell me what you see in $`img`', output='I can see...'),
       mk('note', 'lets add an image note'),
       mk('prompt', 'Please tell me what you see in the 2 images in the above msg',output='i see two puppies!'),
       mk('prompt', 'Whats variable $`y`?')
       ]

In [ ]:
aim_info = dict(supports_vision=True)

A realistic message mix — prompts with and without preceding notes and code, an empty AI response, media references — checks the turn structure:

In [ ]:
*hist,p,_ = msgs2hist(raw, aim_info)

Lets take a look at the history:

In [ ]:
for idx, turn in enumerate(mk_msgs(hist)):
    display(Markdown(f"**======turn: {idx}======**"))
    display(turn)

<div class="prose" markdown="1">

**======turn: 0======**

</div>

<div class="prose" markdown="1">

**Msg**

- role: `user`

<contents>

**Part** (`text`)

<instructions>Respond to the request in the `prompt` below.</instructions>
<prompt sid="_261f0614">hello $`x`</prompt>

<details markdown='1'>

- data: `None`

</details>

</contents>

</div>

<div class="prose" markdown="1">

**======turn: 1======**

</div>

<div class="prose" markdown="1">

**Msg**

- role: `assistant`

<contents>

**Part** (`text`)

Hello! I can see the variable `x`

<details markdown='1'>

- data: `None`

</details>

</contents>

</div>

<div class="prose" markdown="1">

**======turn: 2======**

</div>

...

<div class="prose" markdown="1">

**Msg**

- role: `assistant`

<contents>

**Part** (`text`)

I can see...

<details markdown='1'>

- data: `None`

</details>

</contents>

</div>

<div class="prose" markdown="1">

**======turn: 10======**

</div>

<div class="prose" markdown="1">

**Msg**

- role: `user`

<contents>

**Part** (`text`)

<media id="/static/puppy.jpg" type="content" mime="image/jpeg"></media>

<details markdown='1'>

- data: `None`

</details>

**Part** (`input_image`)

data:image/jpeg;base64,/9j/4AAQSkZJRgABAQAAAQABAAD/4gxUSUNDX1BST0ZJTEUAAQEAAAxEVUNDTQJAAABtbnRyUkdCI...

<details markdown='1'>

- data: `None`

</details>

**Part** (`text`)

<media id="puppy.jpg" type="content" mime="image/jpeg"></media>

<details markdown='1'>

- data: `None`

</details>

**Part** (`input_image`)

data:image/jpeg;base64,/9j/4AAQSkZJRgABAQAAAQABAAD/4gxUSUNDX1BST0ZJTEUAAQEAAAxEVUNDTQJAAABtbnRyUkdCI...

<details markdown='1'>

- data: `None`

</details>

**Part** (`text`)

<message type="note" id="_b0470f05" time=""><cell-source>lets add some images: ![](/static/puppy.jpg#ai) and  ![](puppy.jpg#ai)</cell-source></message>

<details markdown='1'>

- data: `None`

</details>

**Part** (`text`)

<instructions>Respond to the request in the `prompt` below.</instructions>
<prompt sid="_ac22f801">Please tell me what you see in the 2 images in the above msg</prompt>

<details markdown='1'>

- data: `None`

</details>

</contents>

</div>

<div class="prose" markdown="1">

**======turn: 11======**

</div>

<div class="prose" markdown="1">

**Msg**

- role: `assistant`

<contents>

**Part** (`text`)

i see two puppies!

<details markdown='1'>

- data: `None`

</details>

</contents>

</div>

In [ ]:
p

['<instructions>Respond to the request in the `prompt` below.<system-reminder>**NB**: - We are in learning mode</system-reminder></instructions>\n<prompt sid="_ed872c8b">Whats variable $`y`?</prompt>']

In [ ]:
test_eq(len(hist),12)  # `raw` has 6 prompts which means 12 user+ai turns

In [ ]:
hist[6]

['<instructions>Respond to the request in the `prompt` below.</instructions>\n<prompt sid="_6d8e0682">$`x`, $`y`</prompt>']

In [ ]:
# the last prompt is marked `last`, but core rendering is identical: bare content
test_eq(p, [raw[-1].content])
test_eq(hist[6], [raw[8].content])

In [ ]:
# test each user input for expected length
test_eq(len(hist[0]),1) # prompt w/o preceding msgs
test_eq(len(hist[2]),3) # prompt w preceding note and code
test_eq(len(hist[4]),4) # prompt w preceding notes (2) and code
test_eq(len(hist[6]),1) # prompt w/o preceding msgs
test_eq(len(hist[8]),3)  # [note, code, prompt]
test_eq(len(hist[10]),2)  # [note, prompt]

In [ ]:
# Test each AI output for expected length
for i in (1,3,7,9,11): test(len(hist[i]),0,operator.gt)
test_eq(hist[5],'<output result="pending" reason="incomplete"/>')  # empty ai response

## export -

In [ ]:
#| hide
from nbdev import nbdev_export
nbdev_export()